# Medical Triage Classifier — LoRA Fine-Tuning (S3 Edition)

Same training as `train_colab.ipynb`, but pulls data from S3 and uploads
the trained model back to S3. No manual file uploads or downloads.

**Before running:**
1. Set runtime to GPU: **Runtime → Change runtime type → T4 GPU**
2. Run cell 2 to enter your AWS credentials (they stay in memory only — not saved)
3. Run all cells in order
4. Model is uploaded to S3 automatically at the end

## 1. Install Dependencies

In [ ]:
!pip install -q transformers peft accelerate datasets torch scikit-learn boto3

## 2. AWS Credentials

Enter your AWS credentials. These stay in Colab's runtime memory only —
they are **not saved** to the notebook or disk. You'll need to re-enter
them each time the runtime restarts.

In [ ]:
import os
from getpass import getpass

os.environ["AWS_ACCESS_KEY_ID"] = getpass("AWS_ACCESS_KEY_ID: ")
os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("AWS_SECRET_ACCESS_KEY: ")
os.environ["AWS_REGION"] = input("AWS_REGION (default us-east-1): ") or "us-east-1"
os.environ["S3_BUCKET_NAME"] = input("S3_BUCKET_NAME: ")

print(f"\nConfigured: bucket={os.environ['S3_BUCKET_NAME']}, region={os.environ['AWS_REGION']}")

## 3. Download Data from S3

In [ ]:
import io
import boto3
import pandas as pd

S3_PREFIX = "medical-triage-classifier/datasets"
bucket = os.environ["S3_BUCKET_NAME"]

s3 = boto3.client(
    "s3",
    region_name=os.environ["AWS_REGION"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

def load_csv_from_s3(split):
    key = f"{S3_PREFIX}/{split}.csv"
    response = s3.get_object(Bucket=bucket, Key=key)
    df = pd.read_csv(io.StringIO(response["Body"].read().decode("utf-8")))
    print(f"  {split}: {len(df)} rows from s3://{bucket}/{key}")
    return df

print("Downloading data from S3...")
train_df = load_csv_from_s3("train")
val_df = load_csv_from_s3("val")

print(f"\nTrain class distribution:")
print(train_df["urgency"].value_counts().to_string())

## 4. Verify GPU

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {gpu_mem:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

## 5. Tokenize Data

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
LABEL_MAP = {"Routine": 0, "Urgent": 1, "Emergency": 2}
ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
NUM_LABELS = len(LABEL_MAP)
MAX_SEQ_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TriageDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

def tokenize(df):
    encodings = tokenizer(
        df["text"].tolist(),
        truncation=True,
        padding="max_length",
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    )
    labels = torch.tensor(df["urgency"].map(LABEL_MAP).tolist(), dtype=torch.long)
    return TriageDataset(encodings, labels)

train_dataset = tokenize(train_df)
val_dataset = tokenize(val_df)
print(f"Tokenization complete. Vocab size: {tokenizer.vocab_size}")

## 6. Build Model + LoRA Adapters

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

EPOCHS = 5
LEARNING_RATE = 3e-4
BATCH_SIZE = 16
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_MAP,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_lin", "v_lin"],
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.to(device)
model.print_trainable_parameters()

## 7. Train (with class weights)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

# Class weights to prevent majority-class collapse
train_labels = train_df["urgency"].map(LABEL_MAP).values
weights = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float32)
print(f"Class weights: { {ID_TO_LABEL[i]: round(w, 3) for i, w in enumerate(weights)} }")

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=10,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print(f"Training on {device} for {EPOCHS} epochs...")
print(f"Hyperparams: lr={LEARNING_RATE}, batch={BATCH_SIZE}, lora_r={LORA_R}, lora_alpha={LORA_ALPHA}")
print()

train_result = trainer.train()

## 8. Evaluate

In [ ]:
eval_metrics = trainer.evaluate()

print("=== Validation Results ===")
print(f"Accuracy:  {eval_metrics['eval_accuracy']:.4f}")
print(f"F1 Macro:  {eval_metrics['eval_f1_macro']:.4f}")
print(f"Loss:      {eval_metrics['eval_loss']:.4f}")
print(f"\nTrain loss: {train_result.training_loss:.4f}")
print(f"Train time: {train_result.metrics.get('train_runtime', 0):.1f}s")

## 9. Save Model Locally

In [ ]:
SAVE_DIR = "model_final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model saved to {SAVE_DIR}/")
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:40s} {size / 1024:.1f} KB")

## 10. Quick Smoke Test

In [ ]:
from peft import PeftModel

test_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
test_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=ID_TO_LABEL, label2id=LABEL_MAP,
)
test_model = PeftModel.from_pretrained(test_base, SAVE_DIR)
test_model.to(device)
test_model.eval()

samples = [
    ("Patient presents with acute chest pain radiating to left arm. Diaphoretic, BP 180/110.", "Emergency"),
    ("45-year-old female for annual wellness exam. No acute complaints. Well-controlled HTN.", "Routine"),
    ("Worsening abdominal pain x3 days, RLQ, low-grade fever 100.4F. Rebound tenderness.", "Urgent"),
]

print("=== Smoke Test ===")
for text, expected in samples:
    inputs = test_tokenizer(text, truncation=True, padding="max_length",
                            max_length=MAX_SEQ_LENGTH, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = test_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).squeeze()
    pred_id = torch.argmax(probs).item()
    pred_label = ID_TO_LABEL[pred_id]
    confidence = probs[pred_id].item()
    match = "OK" if pred_label == expected else "MISMATCH"
    print(f"  [{match}] Expected: {expected:10s} | Predicted: {pred_label:10s} ({confidence:.1%})")
    print(f"         Scores: {', '.join(f'{ID_TO_LABEL[i]}: {probs[i].item():.1%}' for i in range(NUM_LABELS))}")

## 11. Upload Model to S3

Uploads the trained model to `s3://<bucket>/medical-triage-classifier/models/`.
Download it locally later with:
```bash
aws s3 sync s3://<bucket>/medical-triage-classifier/models/latest/ model_artifacts/final/
```

In [ ]:
from datetime import datetime

S3_MODEL_PREFIX = "medical-triage-classifier/models"
timestamp = datetime.utcnow().strftime("%Y%m%d-%H%M%S")

# Upload to both a timestamped path and a "latest" path
for tag in [f"run-{timestamp}", "latest"]:
    s3_path = f"{S3_MODEL_PREFIX}/{tag}"
    print(f"\nUploading to s3://{bucket}/{s3_path}/")
    for fname in os.listdir(SAVE_DIR):
        filepath = os.path.join(SAVE_DIR, fname)
        if os.path.isfile(filepath):
            key = f"{s3_path}/{fname}"
            s3.upload_file(filepath, bucket, key)
            size_kb = os.path.getsize(filepath) / 1024
            print(f"  {fname:40s} ({size_kb:.1f} KB)")

print(f"\n=== Upload complete ===")
print(f"Timestamped: s3://{bucket}/{S3_MODEL_PREFIX}/run-{timestamp}/")
print(f"Latest:      s3://{bucket}/{S3_MODEL_PREFIX}/latest/")
print(f"\nTo download locally:")
print(f"  aws s3 sync s3://{bucket}/{S3_MODEL_PREFIX}/latest/ model_artifacts/final/")

## 12. (Optional) Also download the zip

If you also want a local copy right now:

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("model_final", "zip", ".", SAVE_DIR)
print(f"Created model_final.zip ({os.path.getsize('model_final.zip') / 1024 / 1024:.1f} MB)")
files.download("model_final.zip")